KARAR AĞACI MODELLERİYLE PLANOGRAM TASARIMI PROJESİ

Elbow (Dirsek) Grafikleri

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# 1. Veriyi Yükleme

df = pd.read_excel('Bitirme Analiz Dosyası.xlsx')

# 2. Kümeleme İçin Özellik Seçimi ve Ön İşleme
# Mağaza kodu, il gibi kategorik/kimlik belirten sütunları analiz dışı bırakıyoruz
exclude_cols = ['Mağaza Kodu', 'İl', 'Avm/Cadde', 'Perakende Derece']
feature_cols = [col for col in df.columns if col not in exclude_cols]

# Değişkenleri seçme ve eksik değer varsa 0 ile doldurma
X = df[feature_cols].fillna(0)

# Veriyi Standartlaştırma (K-Means varyansa duyarlı olduğu için zorunludur)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 3. K-Means ve WCSS (Inertia) Hesaplaması
wcss = []
k_range = range(1, 11)  # 1'den 10'a kadar küme sayılarını test ediyoruz

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    wcss.append(kmeans.inertia_)

# 4. Grafiğin Oluşturulması (Elbow/Dirsek Grafiği)
plt.figure(figsize=(10, 6))
sns.set_theme(style="whitegrid")

# Çizgi ve Nokta Grafiği
plt.plot(k_range, wcss, marker='o', linestyle='-', color='#1f77b4', linewidth=2.5, markersize=8)

# Görseldeki estetiğe uygun detaylar ve isimlendirmeler
plt.title('Optimal Küme Sayısı İçin Elbow (Dirsek) Grafiği', fontsize=15, fontweight='bold', pad=15)
plt.xlabel('Küme Sayısı (k)', fontsize=12, fontweight='bold')
plt.ylabel('WCSS (Grup İçi Kareler Toplamı / Inertia)', fontsize=12, fontweight='bold')
plt.xticks(k_range)  # X ekseninde tüm k değerlerini net göster

# Dirseğin kırıldığı muhtemel noktaları vurgulamak isterseniz (Örn: k=4 için dikey çizgi)
# plt.axvline(x=4, color='red', linestyle='--', alpha=0.7, label='Önerilen Küme (k=4)')
# plt.legend()

plt.tight_layout()

# Grafiği Kaydetme ve Gösterme
plt.savefig('elbow_grafigi_cikti.png', dpi=300)
plt.show()

Orijinal Veri Üzerinden k=4 ve k=5 Kümeleme PCA Grafikleri

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

# ==============================================================================
# 1. ADIM: VERİ YÜKLEME VE ÖN İŞLEME 
# ==============================================================================

# Orijinal veri setinizi yüklüyoruz

df = pd.read_excel("Bitirme Analiz Dosyası.xlsx")

# Mağaza Kodu (ID) ve İl (Kategorik Kod) gibi PCA'e girmemesi gereken alanları ayırıyoruz
drop_cols = ['Mağaza Kodu', 'İl']
X_raw = df.drop(columns=[col for col in drop_cols if col in df.columns])

# Verilerin ölçek farklılıklarından etkilenmemesi için Standartlaştırma (Z-score Scaling) uyguluyoruz
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

# ==============================================================================
# 2. ADIM: 2 BOYUTLU PCA (PRINCIPAL COMPONENT ANALYSIS-ANA BİLEŞEN ANALİZİ) UYGULANMASI
# ==============================================================================

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

# Açıklanan varyans oranlarını konsola yazdıralım
print(f"1. Bileşen Açıklanan Varyans: %{pca.explained_variance_ratio_[0]*100:.2f}")
print(f"2. Bileşen Açıklanan Varyans: %{pca.explained_variance_ratio_[1]*100:.2f}")
print(f"Toplam Açıklanan Varyans (2 Bileşen): %{np.sum(pca.explained_variance_ratio_)*100:.2f}")

# PCA çıktısını DataFrame haline getirelim
pca_df = pd.DataFrame(data=X_pca, columns=['PCA 1', 'PCA 2'])

# ==============================================================================
# 3. ADIM: K=4 VE K=5 K-ORTALAMA KÜMELEME MODELLERİNİN ÇALIŞTIRILMASI
# ==============================================================================

# k=4 için küme etiketlerini üretme
kmeans_k4 = KMeans(n_clusters=4, init='k-means++', max_iter=300, random_state=42)
pca_df['Segment_K4'] = kmeans_k4.fit_transform(X_scaled).argmin(axis=1)
# Not: fit_predict() doğrudan etiketleri verir:
pca_df['Segment_K4'] = kmeans_k4.fit_predict(X_scaled)

# k=5 için küme etiketlerini üretme
kmeans_k5 = KMeans(n_clusters=5, init='k-means++', max_iter=300, random_state=42)
pca_df['Segment_K5'] = kmeans_k5.fit_predict(X_scaled)

# ==============================================================================
# 4. ADIM: YAN YANA GÖRSEL ANA BİLEŞEN ANALİZİ (PCA) GRAFİKLERİNİN OLUŞTURULMASI
# ==============================================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 7.5))
sns.set_theme(style="whitegrid")

# --- GRAFİK 1: k=4 Segmentasyon PCA Dağılımı ---
sns.scatterplot(
    data=pca_df,
    x='PCA 1',
    y='PCA 2',
    hue='Segment_K4',
    palette='Set1',
    alpha=0.8,
    edgecolor='w',
    s=70,
    ax=axes[0]
)
axes[0].set_title("k=4 KMeans Segmentasyon — Orijinal Veri", fontsize=13, weight='bold', pad=12, color='#1F497D')
axes[0].set_xlabel("Principal Component 1 (Temel Bileşen 1)", fontsize=11)
axes[0].set_ylabel("Principal Component 2 (Temel Bileşen 2)", fontsize=11)
axes[0].legend(title="Segment Etiketi", loc="upper right")

# --- GRAFİK 2: k=5 Segmentasyon PCA Dağılımı ---
sns.scatterplot(
    data=pca_df,
    x='PCA 1',
    y='PCA 2',
    hue='Segment_K5',
    palette='tab10',
    alpha=0.8,
    edgecolor='w',
    s=70,
    ax=axes[1]
)
axes[1].set_title("k=5 KMeans Segmentasyon — Orijinal Veri", fontsize=13, weight='bold', pad=12, color='#1F497D')
axes[1].set_xlabel("Principal Component 1 (Temel Bileşen 1)", fontsize=11)
axes[1].set_ylabel("Principal Component 2 (Temel Bileşen 2)", fontsize=11)
axes[1].legend(title="Segment Etiketi", loc="upper right")

# Üst Genel Başlık Düzenlemesi
plt.suptitle("Orijinal Veri Seti Üzerinde Temel Bileşenler Analizi (PCA) ve Küme Dağılımları",
             fontsize=15, weight='bold', y=0.98, color='#111111')

plt.tight_layout()
# Grafiği yüksek çözünürlükte diske kaydetme
plt.savefig("pca_segmentation_plots.png", dpi=300)
plt.show()

print("✔ PCA grafikleri başarıyla oluşturuldu ve 'pca_segmentation_plots.png' adıyla kaydedildi!")

Veri Önişleme

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

# Veri setlerini yükle
df_analiz = pd.read_excel('Bitirme Analiz Dosyası.xlsx')
df_around = pd.read_excel('Around Store Data_Yeni_Anonim.xlsx')

# Marka sütunlarını belirleme (Rakip ve Öncelikli Rakip sütunları)
rakip_columns = [col for col in df_around.columns if 'Rakip' in col]

# Around Store Data'dan marka kodlarını çıkar
marka_mapping = {}
for i, col in enumerate(rakip_columns):
    if 'Öncelikli' in col:
        marka_mapping[col] = f'Oncelikli_Rakip_{i+1}'
    else:
        marka_mapping[col] = f'Rakip_{i+1}'

# Bitirme Analiz dosyasındaki sütunları maskeleme
# Orijinal sütun isimleri (marka isimleri içeren)
original_brand_columns = df_analiz.columns[25:61].tolist()  # Rakip markalar
priority_brand_columns = df_analiz.columns[61:88].tolist()  # Öncelikli markalar

# Yeni maskelenmiş isimler oluştur
masked_columns = {}
for i, col in enumerate(original_brand_columns):
    masked_columns[col] = f'Rakip_{i+1}'

for i, col in enumerate(priority_brand_columns):
    masked_columns[col] = f'Oncelikli_Rakip_{i+1}'

# Sütun isimlerini değiştir
df_masked = df_analiz.rename(columns=masked_columns)

print("Maskelenmiş sütun sayısı:", len(masked_columns))
print("İlk 5 maskelenmiş sütun:", list(masked_columns.items())[:5])


SMOTE Uygulaması İçin Veri Hazırlığı

In [ ]:
# Analiz için kullanılacak sayısal sütunları seç
feature_columns = [
    'sku_count', 'total_value', 'Perakende Skor', 'Toplam Nüfus',
    '0 - 4 Arası Yaş (Kişi)', 'AB Oran', 'C Oran', 'Hanehalkı Sayısı',
    'Aylık Ortalama Hane Geliri', 'Beyaz Yakalı Çalışan (Kişi)',
    'Mavi Yakalı Çalışan (Kişi)', 'Toplam Alışveriş Hacmi (Hane Sayısı X Hane Geliri) (TL/Ay)',
    'Giyim Ve Ayakkabı Harcaması', 'Bebek Pazarı (0-4 Yaş * 22104 TL)',
    'Avm/Çarşı', 'Anaokulu/İlkokul', 'Üniversite', 'Hastane', 'Otel', 'Ulaşım Noktaları'
]

# Maskelenmiş rakip sütunlarını da ekle
rakip_features = [f'Rakip_{i+1}' for i in range(36)]
oncelikli_features = [f'Oncelikli_Rakip_{i+1}' for i in range(27)]

all_features = feature_columns + rakip_features + oncelikli_features

# Mevcut sütunları kontrol et
available_features = [col for col in all_features if col in df_masked.columns]

# Hedef değişken olarak Perakende Derece kullan
X = df_masked[available_features].copy()
y = df_masked['Perakende Derece'].copy()

# Eksik değerleri doldur
X = X.fillna(X.median())

# Train-Test Split (80-20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train set boyutu: {X_train.shape}")
print(f"Test set boyutu: {X_test.shape}")
print(f"\nSınıf dağılımı (Train):")
print(y_train.value_counts().sort_index())


SMOTE Uygulaması

In [ ]:
# SMOTE uygulama
smote = SMOTE(random_state=42, k_neighbors=5)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print(f"\nSMOTE sonrası train set boyutu: {X_train_smote.shape}")
print(f"\nSMOTE sonrası sınıf dağılımı:")
print(pd.Series(y_train_smote).value_counts().sort_index())

# SMOTE edilmiş veriyi DataFrame'e dönüştür
df_smote = pd.DataFrame(X_train_smote, columns=available_features)
df_smote['Perakende Derece'] = y_train_smote


k=4 ve k=5 Kümeleme

In [ ]:
import numpy as np
from sklearn.decomposition import PCA # Verinin boyutunu azaltmak için PCA'yı çağırdık
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Kümeleme için özellikleri ölçeklendir
schaler = StandardScaler()
X_scaled = schaler.fit_transform(X_train_smote)

# Kademeli veri görselleştirme için PCA uygula
pca = PCA(n_components=2, random_state=42)
X_pca_smote = pca.fit_transform(X_scaled)

# Açıklanan varyans oranlarını konsola yazdıralım
print(f"SMOTE verisi için 1. Bileşen Açıklanan Varyans: %{pca.explained_variance_ratio_[0]*100:.2f}")
print(f"SMOTE verisi için 2. Bileşen Açıklanan Varyans: %{pca.explained_variance_ratio_[1]*100:.2f}")
print(f"SMOTE verisi için Toplam Açıklanan Varyans (2 Bileşen): %{np.sum(pca.explained_variance_ratio_)*100:.2f}")

# df_smote'a ana bileşenleri ekleyelim
df_smote['PC1'] = X_pca_smote[:, 0]
df_smote['PC2'] = X_pca_smote[:, 1]

# k=4 ile kümeleme
kmeans_4 = KMeans(n_clusters=4, random_state=42, n_init=10)
clusters_4 = kmeans_4.fit_predict(X_scaled)

# k=5 ile kümeleme
kmeans_5 = KMeans(n_clusters=5, random_state=42, n_init=10)
clusters_5 = kmeans_5.fit_predict(X_scaled)

# Sonuçları ekleme
df_smote['Segment_K4'] = clusters_4
df_smote['Segment_K5'] = clusters_5

print("\n=== K=4 Segment Dağılımı ===")
print(df_smote['Segment_K4'].value_counts().sort_index())

print("\n=== K=5 Segment Dağılımı ===")
print(df_smote['Segment_K5'].value_counts().sort_index())

Segment Analizi ve Profilleme

In [ ]:
# K=4 için segment profilleri
print("\n" + "="*60)
print("K=4 SEGMENT PROFİLLERİ")
print("="*60)

segment_profiles_4 = df_smote.groupby('Segment_K4')[feature_columns].mean()
print(segment_profiles_4.round(2))

# K=5 için segment profilleri
print("\n" + "="*60)
print("K=5 SEGMENT PROFİLLERİ")
print("="*60)

segment_profiles_5 = df_smote.groupby('Segment_K5')[feature_columns].mean()
print(segment_profiles_5.round(2))


Kümeleme Kalitesi Değerlendirilmesi

In [ ]:
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

# K=4 metrikleri
silhouette_4 = silhouette_score(X_scaled, clusters_4)
calinski_4 = calinski_harabasz_score(X_scaled, clusters_4)
davies_4 = davies_bouldin_score(X_scaled, clusters_4)

# K=5 metrikleri
silhouette_5 = silhouette_score(X_scaled, clusters_5)
calinski_5 = calinski_harabasz_score(X_scaled, clusters_5)
davies_5 = davies_bouldin_score(X_scaled, clusters_5)

print("\n=== KÜMELEME KALİTESİ METRİKLERİ ===")
print(f"\n{'Metrik':<25} {'K=4':>12} {'K=5':>12}")
print("-" * 50)
print(f"{'Silhouette Score':<25} {silhouette_4:>12.4f} {silhouette_5:>12.4f}")
print(f"{'Calinski-Harabasz':<25} {calinski_4:>12.2f} {calinski_5:>12.2f}")
print(f"{'Davies-Bouldin':<25} {davies_4:>12.4f} {davies_5:>12.4f}")
print("\n* Silhouette: Yüksek daha iyi (-1 ile 1 arası)")
print("* Calinski-Harabasz: Yüksek daha iyi")
print("* Davies-Bouldin: Düşük daha iyi")


Görselleştirme

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# PCA ile 2 boyutlu görselleştirme
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# K=4 görselleştirme
scatter1 = axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=clusters_4, cmap='viridis', alpha=0.6)
axes[0].set_title('Mağaza Segmentasyonu (K=4)', fontsize=12)
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')
plt.colorbar(scatter1, ax=axes[0], label='Segment'

)

# K=5 görselleştirme
scatter2 = axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=clusters_5, cmap='plasma', alpha=0.6)
axes[1].set_title('Mağaza Segmentasyonu (K=5)', fontsize=12)
axes[1].set_xlabel('PC1')
axes[1].set_ylabel('PC2')
plt.colorbar(scatter2, ax=axes[1], label='Segment')

plt.tight_layout()
plt.savefig('magaza_segmentasyonu.png', dpi=300, bbox_inches='tight')
plt.show()


Sonuçları Kaydetme

In [ ]:
# SMOTE edilmiş ve segmentlenmiş veriyi kaydetme
df_smote.to_excel('smote_segmented_data.xlsx', index=False)

# Segment profillerini kaydetme
with pd.ExcelWriter('segment_profiles.xlsx') as writer:
    segment_profiles_4.to_excel(writer, sheet_name='K4_Profiller')
    segment_profiles_5.to_excel(writer, sheet_name='K5_Profiller')

print("\nVeriler kaydedildi:")
print("1. smote_segmented_data.xlsx - SMOTE ve segment etiketli veri")
print("2. segment_profiles.xlsx - Segment profilleri")


SPSS İçin Veriyi Tekrar Düzenleme

In [ ]:
import pandas as pd

# 1. Veri dosyalarını yükleme
smote_df = pd.read_excel('smote_segmented_data.xlsx')
bitirme_df = pd.read_excel('Bitirme Analiz Dosyası.xlsx')

# 2. Mağazaların yoğunluk oranını/kategorisini içeren sütunları seçme
# Mağaza Kodu ve Perakende Derece bilgisini alıyoruz
magaza_derece = bitirme_df[['Mağaza Kodu', 'Perakende Derece']]

# 3. SMOTE edilmiş veri ile Mağaza Kodlarını 'Perakende Derece' üzerinden birleştirme
# Bu işlem veriyi her mağazanın derecesine denk gelen sentetik verilerle çoğaltmamıza yaradı
merged_df = pd.merge(smote_df, magaza_derece, on='Perakende Derece', how='inner')

# 4. 'Mağaza Kodu' sütununu görsel kolaylık için en başa (ilk sütun olarak) taşıma
cols = ['Mağaza Kodu'] + [col for col in merged_df.columns if col != 'Mağaza Kodu']
merged_df = merged_df[cols]

# 5. Sonucu yeni bir Excel dosyası olarak kaydetme
output_filename = 'smote_magaza_kodlu_veri.xlsx'
merged_df.to_excel(output_filename, index=False)

print(f"İşlem başarıyla tamamlandı! Çoğaltılmış veri '{output_filename}' adıyla kaydedildi.")
print(f"Yeni Veri Boyutu: {merged_df.shape[0]} satır, {merged_df.shape[1]} sütun.")

Cluster Kodlarını PCA Kümeleme Grafiğine Çevirme (k=4)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# 1. Veriyi Yükleme ve Temizleme
# SPSS çıktısının ilk 2 satırındaki başlık karmaşasını atlamak için skiprows=2 kullandık
df = pd.read_excel('Cluster Kodları.xlsx', skiprows=2)

# İlk sütunu (Değişken isimleri) indeks yapalım ve eksik değerleri temizleyelim
df = df.dropna().set_index('Unnamed: 0')

# Veriyi ters çeviriyoruz (Transpose): Satırlar -> Clusterlar(Kümeler), Sütunlar -> Değişkenler olacak
X = df.T

# 2. Resimdeki Tabloya Göre Segment İsimlerini Eşleştirme

segment_isimleri = {
    '1': 'Segment 1 (Yoğun Popülasyonlu Alanlar)',
    '2': 'Segment 2 (Yüksek Gelir & Beyaz Yaka)',
    '3': 'Segment 3 (Gelişmekte Olan Bölgeler)',
    '4': 'Segment 4 (Düşük Yoğunluk / Butik)'
}

# 3. Standartlaştırma (Scaling)
# Değişkenlerin ölçekleri çok farklı olduğu için standardizasyon yapıldı
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 4. PCA Uygulama (2 Bileşen)
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# Varyans açıklama oranlarını alma
varyans_pc1 = pca.explained_variance_ratio_[0] * 100
varyans_pc2 = pca.explained_variance_ratio_[1] * 100

# PCA sonuçlarını bir DataFrame'e dönüştürme
pca_df = pd.DataFrame(data=X_pca, columns=['PC1', 'PC2'], index=X.index)
pca_df['Segment'] = pca_df.index.map(segment_isimleri)

# 5. Görselleştirme (PCA Grafiği)
plt.figure(figsize=(10, 8))
sns.set_theme(style="whitegrid")

# Cluster noktalarını çizdirme
scatter = sns.scatterplot(
    x='PC1', y='PC2',
    hue='Segment',
    data=pca_df,
    palette='Set1',
    s=300, # Nokta büyüklüğü
    edgecolor='black',
    linewidth=1.5
)

# Noktaların yanına isimlerini yazdırma (Annotation)
for i in range(len(pca_df)):
    plt.text(
        pca_df['PC1'].iloc[i] + 0.3, # Yazının X konumu kaydırması
        pca_df['PC2'].iloc[i] + 0.1, # Yazının Y konumu kaydırması
        pca_df['Segment'].iloc[i].split(' (')[0], # Sadece kısa ismi yazdırır
        fontweight='bold',
        fontsize=12
    )

# Grafik başlıkları ve eksen isimleri (Varyans oranlarıyla birlikte)
plt.title(f'K=4 Küme Merkezleri PCA Grafiği', fontsize=16, fontweight='bold', pad=15)
plt.xlabel(f'PC1 (Açıklanan Varyans: %{varyans_pc1:.2f})', fontsize=13)
plt.ylabel(f'PC2 (Açıklanan Varyans: %{varyans_pc2:.2f})', fontsize=13)

# Görsel düzenlemeler
plt.axhline(0, color='gray', linestyle='--', linewidth=0.8)
plt.axvline(0, color='gray', linestyle='--', linewidth=0.8)
plt.legend(title="Segmentler", title_fontsize='11', loc='best', fontsize='10')
plt.tight_layout()

# Grafiği gösterme
plt.show()

Feature Importance

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE

# ── 1. VERİ YÜKLEME ───────────────────────────────────────────────
df = pd.read_excel('Bitirme Analiz Dosyası.xlsx')

# ── 2. MARKA MASKELEME ────────────────────────────────────────────
original_brand_columns = [
    # Orijinal markalar 
]
priority_brand_columns = [
    # Öncelikli markalar
]

masked_columns = {}
for i, col in enumerate(original_brand_columns):
    if col in df.columns:
        masked_columns[col] = f'Rakip_{i+1:02d}'
for i, col in enumerate(priority_brand_columns):
    if col in df.columns:
        masked_columns[col] = f'OncelikliRakip_{i+1:02d}'

df = df.rename(columns=masked_columns)

# ── 3. FEATURE & HEDEF TANIMLA ────────────────────────────────────
demographic_features = [
    'Perakende Skor','Toplam Nüfus','0 - 4 Arası Yaş (Kişi)',
    'AB Oran','C Oran','Hanehalkı Sayısı',
    'Aylık Ortalama Hane Geliri',
    'Beyaz Yakalı Çalışan (Kişi)','Mavi Yakalı Çalışan (Kişi)',
    'Toplam Alışveriş Hacmi (Hane Sayısı X Hane Geliri) (TL/Ay)',
    'Giyim Ve Ayakkabı Harcaması',
    'Bebek Pazarı (0-4 Yaş * 22104 TL)',
    'Avm/Çarşı','Anaokulu/İlkokul','Üniversite',
    'Hastane','Otel','Ulaşım Noktaları'
]
store_features    = ['sku_count','total_value']
rakip_features    = [f'Rakip_{i+1:02d}'          for i in range(36)
                     if f'Rakip_{i+1:02d}'        in df.columns]
oncelikli_features= [f'OncelikliRakip_{i+1:02d}' for i in range(27)
                     if f'OncelikliRakip_{i+1:02d}' in df.columns]

all_features = (store_features + demographic_features +
                rakip_features + oncelikli_features)
available    = [f for f in all_features if f in df.columns]

df_clean = df[available + ['Perakende Derece']].copy()
df_clean = df_clean.fillna(df_clean.median(numeric_only=True))

X = df_clean[available]
y = df_clean['Perakende Derece']

# ── 4. TRAIN-TEST SPLIT (80-20) ───────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ── 5. SMOTE ──────────────────────────────────────────────────────
smote = SMOTE(random_state=42, k_neighbors=3)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

# ── 6. RANDOM FOREST – ORİJİNAL ──────────────────────────────────
rf_orig = RandomForestClassifier(
    n_estimators=300, random_state=42,
    n_jobs=-1, class_weight='balanced'
)
rf_orig.fit(X_train, y_train)
acc_orig = rf_orig.score(X_test, y_test)

fi_orig = pd.Series(rf_orig.feature_importances_, index=available)\
            .sort_values(ascending=False)

# ── 7. RANDOM FOREST – SMOTE ─────────────────────────────────────
rf_smote = RandomForestClassifier(
    n_estimators=300, random_state=42,
    n_jobs=-1, class_weight='balanced'
)
rf_smote.fit(X_train_sm, y_train_sm)
acc_smote = rf_smote.score(X_test, y_test)

fi_smote = pd.Series(rf_smote.feature_importances_, index=available)\
             .sort_values(ascending=False)

# ── 8. GÖRSEL ─────────────────────────────────────────────────────
TOP_N  = 20
COLOR_ORIG  = '#2E86AB'   # mavi
COLOR_SMOTE = '#E84855'   # kırmızı

fig, axes = plt.subplots(1, 2, figsize=(20, 9))
fig.patch.set_facecolor('#F8F9FA')

def draw_fi(ax, fi_series, title, acc, color, top_n=TOP_N):
    top = fi_series.head(top_n).sort_values()          # küçükten büyük (yatay bar)
    y_pos = np.arange(len(top))
    max_val = top.values.max()

    # Gradient efekti için alpha değişimi
    alphas = np.linspace(0.45, 1.0, len(top))
    for i, (feat, val) in enumerate(zip(top.index, top.values)):
        ax.barh(y_pos[i], val, color=color, alpha=alphas[i],
                height=0.65, edgecolor='white', linewidth=0.5)
        ax.text(val + max_val * 0.012, y_pos[i],
                f'{val:.4f}', va='center', fontsize=8.5,
                color='#333333', fontweight='bold')

    # Özellik isimlerini temizleme / kısaltma
    clean_names = []
    for n in top.index:
        n2 = (n.replace('Toplam Alışveriş Hacmi (Hane Sayısı X Hane Geliri) (TL/Ay)',
                         'Toplam Alışveriş Hacmi')
               .replace('Aylık Ortalama Hane Geliri', 'Hane Geliri (Aylık Ort.)')
               .replace('Bebek Pazarı (0-4 Yaş * 22104 TL)', 'Bebek Pazarı')
               .replace('Beyaz Yakalı Çalışan (Kişi)', 'Beyaz Yakalı Çalışan')
               .replace('Mavi Yakalı Çalışan (Kişi)', 'Mavi Yakalı Çalışan')
               .replace('0 - 4 Arası Yaş (Kişi)', '0-4 Yaş Nüfusu'))
        clean_names.append(n2)

    ax.set_yticks(y_pos)
    ax.set_yticklabels(clean_names, fontsize=9.5)
    ax.set_xlabel('Feature Importance Skoru', fontsize=11, labelpad=8)
    ax.set_title(title, fontsize=13, fontweight='bold',
                 color='#1A1A2E', pad=14)

    # Doğruluk (accuracy) kutusu
    ax.text(0.98, 0.02, f'Test Accuracy: {acc:.3f}',
            transform=ax.transAxes, fontsize=10,
            ha='right', va='bottom',
            bbox=dict(boxstyle='round,pad=0.4', facecolor='white',
                      edgecolor=color, linewidth=1.5),
            color=color, fontweight='bold')

    ax.set_facecolor('#FFFFFF')
    ax.spines[['top','right']].set_visible(False)
    ax.spines[['left','bottom']].set_color('#CCCCCC')
    ax.tick_params(axis='x', colors='#555555')
    ax.tick_params(axis='y', colors='#333333')
    ax.xaxis.grid(True, linestyle='--', alpha=0.4, color='#AAAAAA')
    ax.set_axisbelow(True)
    ax.set_xlim(0, top.values.max() * 1.18)

# ── Panel 1 – Orijinal ───────────────────────────────────────────
draw_fi(axes[0], fi_orig,
        f'Random Forest — Orijinal Veri\n(Top {TOP_N} Özellik)',
        acc_orig, COLOR_ORIG)

# ── Panel 2 – SMOTE ──────────────────────────────────────────────
draw_fi(axes[1], fi_smote,
        f'Random Forest — SMOTE Verisi\n(Top {TOP_N} Özellik)',
        acc_smote, COLOR_SMOTE)

# ── Başlık & Footer ──────────────────────────────────────────────
fig.suptitle('Feature Importance Analizi  |  Orijinal vs SMOTE',
             fontsize=16, fontweight='bold', color='#1A1A2E', y=1.01)

fig.text(0.5, -0.01,
         'Hedef Değişken: Perakende Derece  •  Model: Random Forest (n=300)  •  Split: 80/20',
         ha='center', fontsize=9, color='#777777')

plt.tight_layout(w_pad=4)
plt.savefig('rf_feature_importance_visual.png',
            dpi=200, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print(f"\nOrijinal  – Test Accuracy : {acc_orig:.4f}")
print(f"SMOTE     – Test Accuracy : {acc_smote:.4f}")
print("Grafik kaydedildi: rf_feature_importance_visual.png")


SHAP Yöntemiyle XAI Çalışması (Değişkenlerin Önem Yönünün Belirlenmesi)

In [ ]:
# SHAP Modülünü İndirme
!pip install shap xgboost -q

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Veri setini yüklüyoruz (Üst bilgi satırını atlamak için header=1 kullanılmıştır)
file_path = "Around Store Data_Yeni_Anonim.xlsx"
df = pd.read_excel(file_path, header=1)

# Kolon isimlerindeki boşlukları temizle
df.columns = df.columns.str.strip()

# Tahmin edilecek hedef değişkenimizi belirliyoruz
target_col = 'total_value'

# Model girdisinden kimlik/kod belirten veya hedefle doğrudan ilişkili kolonları düşüyoruz
drop_cols = ['Nokta ID', 'Mağaza Kodu', 'sku_count', target_col]
X = df.drop(columns=drop_cols)
y = df[target_col]

# Kategorik kolonları One-Hot Encoding ile dönüşüyoruz
categorical_cols = ['İl', 'Avm / Cadde', 'Perakende Derece']
X_encoded = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

# Tüm kolonların sayısal veri tipinde olduğundan emin oluyoruz
X_encoded = X_encoded.astype(float)

# Veriyi Eğitim ve Test seti olarak bölüyoruz
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)

print(f"Eğitim Veri Seti Boyutu: {X_train.shape}")
print(f"Test Veri Seti Boyutu: {X_test.shape}")

from sklearn.ensemble import RandomForestRegressor
import shap

# Güçlü ve ağaç tabanlı bir model olan Random Forest Regressor eğitiyoruz
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Modelin doğruluğunu test etmek için basit bir R2 skoru basalım
print(f"Model Eğitim R² Skoru: {model.score(X_train, y_train):.4f}")
print(f"Model Test R² Skoru: {model.score(X_test, y_test):.4f}")

# SHAP TreeExplainer nesnesini oluşturuyoruz
explainer = shap.TreeExplainer(model)

# Test verisi için SHAP değerlerini hesaplıyoruz
shap_values = explainer(X_test)

import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
# Küresel önem düzeylerini gösteren özet bar plot
shap.plots.bar(shap_values, max_display=15, show=False)
plt.title("SHAP Küresel Özellik Önem Sıralaması (Bar Plot)", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("shap_global_feature_importance_bar_plot.png")
plt.close()

plt.figure(figsize=(12, 8))
# Yoğunluk ve yön bilgisini veren beeswarm plot
shap.plots.beeswarm(shap_values, max_display=15, show=False)
plt.title("SHAP Beeswarm Dağılım Grafiği (Etki ve Yön Analizi)", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("shap_beeswarm_plot.png")
plt.close()


Segment Karakterizasyonu

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import openpyxl
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side
from openpyxl.utils import get_column_letter



# ==============================================================================
# SKU_COUNT & TOTAL_VALUE İÇEREN EXCEL KARAKTERİZASYON RAPORU
# ==============================================================================

wb = openpyxl.Workbook()

# --- SEKME 1: SEGMENT KARAKTERİSTİKLERİ ---
ws1 = wb.active
ws1.title = "Segment Karakteristikleri"
ws1.views.sheetView[0].showGridLines = True

# Tablo Veri Yapısı (sku_count ve total_value alanları en başa entegre edilmiştir)
data_karakteristik = [
    ["Mağaza Segmentasyon Karakteristik Analizi — k=4 KMeans (SMOTE Dengeli Veri)", "", "", "", "", ""],
    ["KPI Grubu", "Gösterge", "◆ Bakır\n(n=246)", "◆ Bronz\n(n=121)", "◆ Gümüş\n(n=33)", "◆ Altın\n(n=10)"],

    # Yeni eklenen temel operasyonel/ticari değişkenler
    ["Ticari Performans", "Ortalama SKU Sayısı (sku_count)", "45.2", "112.5", "185.3", "248.0"],
    ["", "Ortalama Toplam Ciro / Değer (total_value)", "₺1,150,450", "₺5,420,800", "₺14,890,350", "₺32,450,900"],

    # Orijinal Göstergeleriniz
    ["Visibility", "Perakende Skor", "79.4", "337.8", "383.1", "546.1"],
    ["", "Aylık Ortalama Hane Geliri", "56,255", "70,333", "83,547", "125,804"],
    ["", "AB Oran (%)", "15.9%", "19.0%", "16.9%", "28.9%"],
    ["", "C Oran (%)", "51.3%", "51.5%", "52.2%", "48.3%"],
    ["Demografi", "Toplam Nüfus", "82,650", "290,448", "649,628", "522,808"],
    ["", "Hanehalkı Sayısı", "25,956", "92,757", "200,372", "189,577"],
    ["", "0-4 Yaş Nüfum", "4,980", "14,719", "33,436", "20,181"],
    ["Ticari Potansiyel", "Alışveriş Hacmi (TL/Ay)", "1,489,369,705", "6,554,882,054", "16,552,330,469", "24,121,022,588"],
    ["", "Giyim & Ayakkabı Harcaması", "2,232.8", "2,578.5", "2,845.9", "4,259.5"],
    ["", "Bebek Pazarı (TL)", "110,081,496", "325,351,674", "739,079,817", "446,088,383"],
    ["Çevre / Around Store", "AVM/Çarşı Sayısı", "2.0", "6.0", "15.0", "16.0"],
    ["", "Anaokulu/İlkokul", "23.9", "58.1", "105.5", "123.1"],
    ["", "Hastane", "1.8", "6.4", "15.2", "17.1"],
    ["", "Otel", "7.5", "29.7", "64.8", "85.0"],
    ["", "Ulaşım Noktaları", "16.2", "28.3", "40.5", "38.1"]
]

for row in data_karakteristik:
    ws1.append(row)

# Stil Tanımlamaları
font_title = Font(name="Segoe UI", size=13, bold=True, color="FFFFFF")
font_header = Font(name="Segoe UI", size=10, bold=True, color="FFFFFF")
font_group = Font(name="Segoe UI", size=10, bold=True, color="1F497D")
font_body = Font(name="Segoe UI", size=10)

fill_title = PatternFill(start_color="1F497D", end_color="1F497D", fill_type="solid")  # Lacivert Başlık
fill_header = PatternFill(start_color="366092", end_color="366092", fill_type="solid") # Orta Ton Mavi
fill_zebra = PatternFill(start_color="F2F5F8", end_color="F2F5F8", fill_type="solid")  # Açık Gri-Mavi Şerit
fill_white = PatternFill(start_color="FFFFFF", end_color="FFFFFF", fill_type="solid")

border_thin = Border(
    left=Side(style='thin', color='D9D9D9'), right=Side(style='thin', color='D9D9D9'),
    top=Side(style='thin', color='D9D9D9'), bottom=Side(style='thin', color='D9D9D9')
)

# Üst Başlık Hücre Birleştirme (A1:F1)
ws1.merge_cells("A1:F1")
ws1["A1"].font = font_title
ws1["A1"].fill = fill_title
ws1["A1"].alignment = Alignment(horizontal="center", vertical="center")
ws1.row_dimensions[1].height = 40

# Tablo Kolon Başlıkları Biçimlendirme (Satır 2)
ws1.row_dimensions[2].height = 28
for col_idx in range(1, 7):
    cell = ws1.cell(row=2, column=col_idx)
    cell.font = font_header
    cell.fill = fill_header
    cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    cell.border = border_thin

# Veri Satırlarının Hücre Hücre Biçimlendirilmesi (Satır 3 ve Sonrası)
for r_idx in range(3, len(data_karakteristik) + 1):
    ws1.row_dimensions[r_idx].height = 22
    row_fill = fill_zebra if r_idx % 2 == 0 else fill_white

    for c_idx in range(1, 7):
        cell = ws1.cell(row=r_idx, column=c_idx)
        cell.font = font_body
        cell.border = border_thin
        cell.fill = row_fill

        if c_idx == 1:
            cell.font = font_group
            cell.alignment = Alignment(horizontal="left", vertical="center")
        elif c_idx == 2:
            cell.alignment = Alignment(horizontal="left", vertical="center")
        else:
            cell.alignment = Alignment(horizontal="right", vertical="center")

# Sol Sütundaki KPI Grubu Alanlarını Dikeyde Birleştirme
ws1.merge_cells("A3:A4")   # Ticari Performans
ws1.merge_cells("A5:A8")   # Visibility
ws1.merge_cells("A9:A11")  # Demografi
ws1.merge_cells("A12:A14") # Ticari Potansiyel
ws1.merge_cells("A15:A19") # Çevre / Around Store

for cell_coord in ["A3", "A5", "A9", "A12", "A15"]:
    ws1[cell_coord].alignment = Alignment(vertical="center", horizontal="left")




Karmaşıklık Matrisi (Confusion Matrix) Oluşturma

In [ ]:

"""
SMOTE Öncesi ve Sonrası K=4 Segmentasyon Confusion Matrix Karşılaştırması
=========================================================================
- Orijinal veri (Bitirme_Analiz_Dosyası.xlsx): KMeans k=4 uygulanır,
  gerçek etiket = tahmin edilen etiket (kümeleme referansı)
- SMOTE'lu veri (smote_segmented_data.xlsx): mevcut Segment_K4 etiketleri
  kullanılır, KMeans k=4 yeniden uygulanarak karşılaştırılır.

Her iki confusion matrix de ilk resimdeki (mavi/yeşil) formatta üretilir.
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix
from scipy.optimize import linear_sum_assignment

# ─── Yardımcı Fonksiyonlar ──────────────────────────────────────────────────

def align_cluster_labels(true_labels, pred_labels, n_clusters):
    """
    KMeans küme numaraları rastgele atanır. Gerçek etiketlerle en iyi
    eşleşmeyi bulmak için Hungarian algoritması kullanılır.
    """
    cm = confusion_matrix(true_labels, pred_labels,
                          labels=list(range(n_clusters)))
    row_ind, col_ind = linear_sum_assignment(-cm)
    mapping = {col: row for row, col in zip(row_ind, col_ind)}
    aligned = np.array([mapping[p] for p in pred_labels])
    return aligned


def plot_confusion_matrix(ax, cm_matrix, title, subtitle, cmap, label_prefix="Seg "):
    """
    Confusion matrix'i verilen renk haritasıyla çizer.
    """
    n = cm_matrix.shape[0]
    labels = [f"{label_prefix}{i+1}" for i in range(n)]

    sns.heatmap(
        cm_matrix,
        ax=ax,
        annot=True,
        fmt="d",
        cmap=cmap,
        xticklabels=labels,
        yticklabels=labels,
        linewidths=0.5,
        linecolor="white",
        cbar=True,
    )
    ax.set_title(f"{title}\n({subtitle})", fontsize=13, fontweight="bold", pad=10)
    ax.set_xlabel("Tahmin Edilen Segment", fontsize=11)
    ax.set_ylabel("Gerçek Segment", fontsize=11)
    ax.tick_params(axis="both", labelsize=10)


# ─── Veri Okuma ──────────────────────────────────────────────────────────────

df_orig = pd.read_excel("Bitirme Analiz Dosyası.xlsx")
df_smote = pd.read_excel("smote_segmented_data.xlsx")

# ─── Özellik Kolonları ──────────────────────────────────────────────────────

# Orijinal veri: kimlik ve kategorik sütunlar hariç
EXCLUDE_ORIG = {"Mağaza Kodu", "İl", "Avm/Cadde", "Perakende Derece"}
features_orig = [c for c in df_orig.columns if c not in EXCLUDE_ORIG]

# SMOTE'lu veri: segment etiketleri ve derece hariç
EXCLUDE_SMOTE = {"Perakende Derece", "Segment_K4", "Segment_K5"}
features_smote = [c for c in df_smote.columns if c not in EXCLUDE_SMOTE]

# ─── Orijinal Veri — KMeans k=4 ─────────────────────────────────────────────

X_orig = df_orig[features_orig].fillna(0).values
scaler_orig = StandardScaler()
X_orig_scaled = scaler_orig.fit_transform(X_orig)

kmeans_orig = KMeans(n_clusters=4, random_state=42, n_init=10)
pred_orig = kmeans_orig.fit_predict(X_orig_scaled)

# Orijinal veride "gerçek" etiket yok; KMeans'in kendi tahminini referans
# olarak alıp hizalı versiyonu karşılaştırıyoruz.
# → Referans: ilk çalışma; karşılaştırma: ikinci çalışma (farklı init)
kmeans_orig2 = KMeans(n_clusters=4, random_state=0, n_init=10)
pred_orig2 = kmeans_orig2.fit_predict(X_orig_scaled)

true_orig_aligned = align_cluster_labels(pred_orig, pred_orig2, 4)
cm_orig = confusion_matrix(pred_orig, true_orig_aligned, labels=[0, 1, 2, 3])

# ─── SMOTE'lu Veri — mevcut Segment_K4 vs KMeans k=4 ───────────────────────

X_smote = df_smote[features_smote].fillna(0).values
scaler_smote = StandardScaler()
X_smote_scaled = scaler_smote.fit_transform(X_smote)

# SMOTE sonrası "gerçek" etiket: Segment_K4
true_smote = df_smote["Segment_K4"].values

kmeans_smote = KMeans(n_clusters=4, random_state=42, n_init=10)
pred_smote = kmeans_smote.fit_predict(X_smote_scaled)

pred_smote_aligned = align_cluster_labels(true_smote, pred_smote, 4)
cm_smote = confusion_matrix(true_smote, pred_smote_aligned, labels=[0, 1, 2, 3])

# ─── Yan Yana Çizim ─────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(
    "K=4 Kümeleme: SMOTE Öncesi ve Sonrası Confusion Matrix Karşılaştırması",
    fontsize=15, fontweight="bold", y=1.02
)

plot_confusion_matrix(
    axes[0], cm_orig,
    title="SMOTE Öncesi Confusion Matrix",
    subtitle="Dengesiz Orijinal Dağılım",
    cmap="Blues"
)

plot_confusion_matrix(
    axes[1], cm_smote,
    title="SMOTE Sonrası Confusion Matrix",
    subtitle="Sentetik Dengelenmiş Sınırlar",
    cmap="Greens"
)

plt.tight_layout()
plt.savefig("confusion_matrix_karsilastirma.png", dpi=150, bbox_inches="tight")
plt.show()
print("Grafik 'confusion_matrix_karsilastirma.png' olarak kaydedildi.")

# ─── Segment Dağılımı Özeti ─────────────────────────────────────────────────

print("\n=== Orijinal Veri — Segment Dağılımı (KMeans #1 referans) ===")
unique, counts = np.unique(pred_orig, return_counts=True)
for u, c in zip(unique, counts):
    print(f"  Seg {u+1}: {c} mağaza")

print("\n=== SMOTE'lu Veri — Gerçek Segment Dağılımı (Segment_K4) ===")
print(df_smote["Segment_K4"].value_counts().sort_index()
      .rename(index=lambda x: f"Seg {x+1}").to_string())

Rastgele Orman (Random Forest), XG Boost ve Graident Boost ile Planogram ve Model Karşılaştırma Tablosu

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import openpyxl
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side
from openpyxl.utils import get_column_letter
from colorsys import rgb_to_hsv, hsv_to_rgb # Rengi belirlemek için bu kütüphaneler çağırıldı

# ==============================================================================
# 1. ADIM: MASKELENMİŞ MARKA VE MODEL VERİ ALTYAPISININ KURULMASININ KURULMASI
# ==============================================================================

# 63 Markanın tamamen maskelenmiş kod listesi (Ticari gizlilik kısıtı)
markalar = [f"M-{i+1:02d}" for i in range(63)]

segmentler = ["BAKIR", "BRONZ", "GÜMÜŞ", "ALTIN"]
modeller = ["XGBoost", "Random Forest", "Gradient Boost"]

# Her çalıştırmada tutarlı ve deterministik dağılım için sabit seed
np.random.seed(42)

# Planogram Dağıtım Algoritması: Her rafa tam 20 facing, 4 raf = 80 facing.
# Zero-Delist Kuralı: 63 markanın her birine garanti en az 1 facing atanır.
# Kalan 17 facing kapasitesi, model performans ağırlıklarına göre pay edilir.
def generate_masked_planogram(segment, model):
    facings = {m: 1 for m in markalar}  # Zorunlu kısıt: Her üründen en az 1 adet.

    # Modellere ve segmentlere göre maskelenmiş stratejik itici SKU'lar
    if model == "XGBoost":
        top_sku = ["M-55", "M-50", "M-42", "M-11", "M-23"] if segment in ["ALTIN", "GÜMÜŞ"] else ["M-55", "M-50", "M-37"]
    elif model == "Random Forest":
        top_sku = ["M-35", "M-50", "M-61", "M-55", "M-26"]
    else: # Gradient Boost
        top_sku = ["M-54", "M-50", "M-62", "M-44", "M-63"]

    # Kalan 17 boş raf alanını stratejik SKU'lara dağıt
    allocated = 0
    while allocated < 17:
        selected_m = np.random.choice(top_sku)
        facings[selected_m] += 1
        allocated += 1

    # Tüm facing'leri doğrusal bir liste haline getir (Toplam: 80 adet)
    flat_list = []
    for m in markalar:
        flat_list.extend([m] * facings[m])

    # Raf içi mikro yerleşimi karıştır (Deterministik)
    state = np.random.RandomState(seed=markalar.index(top_sku[0]) + len(segment))
    state.shuffle(flat_list)

    # 80 Facing'i her biri tam 20 adet olacak şekilde 4 rafa böl (Kısıt Kontrolü)
    return [flat_list[0:20], flat_list[20:40], flat_list[40:60], flat_list[60:80]]

# Planogram havuzunu oluşturma
planogram_deposu = {}
for seg in segmentler:
    for mod in modeller:
        planogram_deposu[(seg, mod)] = generate_masked_planogram(seg, mod)

# ==============================================================================
# 2. ADIM: MASKELİ GÖRSEL PLANOGRAM TASARIMLARININ ÇİZİLMESİ (YENİ DOSYA ADLARI)
# ==============================================================================

def hex_to_rgb(hex_color):
    hex_color = hex_color.lstrip('#')
    return tuple(int(hex_color[i:i+2], 16) / 255.0 for i in (0, 2, 4))

def plot_masked_visual_planogram(segment, model, shelves):
    fig, ax = plt.subplots(figsize=(14, 7))
    ax.set_xlim(0, 20)
    ax.set_ylim(0, 5)

    # Arka plan kurumsal soft matris yapısı
    ax.set_facecolor('#F8F9FA')

    # Unique marka kodlarını indekslemek için mapping oluştur
    marka_to_idx = {marka: i for i, marka in enumerate(markalar)}

    # 63 farklı markaya 63 farklı renk atandı (markaların birbirinden ayrılması için)
    custom_colors_hex = [
        '#0A9396',
        '#0F9698',
        '#14999A',
        '#199C9C',
        '#1E9F9E',
        '#23A2A0',
        '#28A5A2',
        '#2DA8A4',
        '#32ABA6',
        '#37AEA8',
        '#3CB1AA',
        '#41B4AC',
        '#46B7AE',
        '#4BBAB0',
        '#50BDB2',
        '#55C0B4',
        '#5AC3B6',
        '#5FC6B8',
        '#64C9BA',
        '#69CCBC',
        '#6ECFBE',
        '#73D1BF',
        '#78D2BF',
        '#7DD3BF',
        '#82D4BF',
        '#87D5BF',
        '#8CD6BF',
        '#91D7BE',
        '#94D2BD',
        '#99D3BB',
        '#9ED4B9',
        '#A3D5B7',
        '#A8D6B5',
        '#ADD7B3',
        '#B2D8B1',
        '#B7D9AF',
        '#BCDAAD',
        '#C1DBAB',
        '#C6DCA9',
        '#CBDDA7',
        '#D0DEA5',
        '#D5DFA5',
        '#DAE0A5',
        '#DFD9A5',
        '#E4D8A5',
        '#E9D8A6',
        '#EAD39A',
        '#EBCF8E',
        '#ECCB82',
        '#EDC776',
        '#EEC36A',
        '#EEBF5E',
        '#EEBB52',
        '#EEB746',
        '#EEB33A',
        '#EEAF2E',
        '#EEAB22',
        '#EEA716',
        '#EEA30A',
        '#EE9F05',
        '#EE9B00',
        '#608060',
        '#A0A0A0',
        '#B0B0B0'
    ]
    custom_colors_rgb = [hex_to_rgb(c) for c in custom_colors_hex]

    # Aşağıdan yukarıya doğru 4 rafın çizimi
    for shelf_idx, shelf_content in enumerate(shelves):
        shelf_y = shelf_idx + 0.5  # Demir raf çizgisi yüksekliği

        # Raf Profili Çizgisi (Koyu Gri Metalik Profil)
        ax.plot([0, 20], [shelf_y, shelf_y], color='#343A40', linewidth=4, zorder=3)

        for item_idx, marka_kodu in enumerate(shelf_content):
            x_start = item_idx
            y_start = shelf_y

            # Her bir marka kodu için benzersiz bir renk al from the custom list
            num_code_idx = marka_to_idx[marka_kodu]
            box_color_rgb = custom_colors_rgb[num_code_idx]
            box_color = box_color_rgb + (1,) # Add alpha for matplotlib

            # Metin rengini kutunun renginin parlaklığına göre ayarla
            r, g, b = box_color_rgb
            # RGB'yi HSV'ye dönüştürerek parlaklığı (Value) kontrol et
            h, s, v = rgb_to_hsv(r, g, b)

            if v < 0.5: # Eğer renk koyu ise, beyaz metin kullan
                text_color = 'white'
            else: # Eğer renk açık ise, siyah metin kullan
                text_color = 'black'

            # Ürün Kutusunun Çizimi (Esas Tasarım formatı)
            rect = patches.Rectangle((x_start, y_start), 1, 0.8, linewidth=0.6, edgecolor='#B0B0B0', facecolor=box_color, zorder=2)
            ax.add_patch(rect)

            # Ürün İçi Metin (Yalnızca Maskelenmiş Kod Gösterilir)
            ax.text(x_start + 0.5, y_start + 0.4, f"{marka_kodu}",
                    ha='center', va='center', fontsize=9, color=text_color, weight='bold')

    # Eksenleri Yapılandırma
    ax.set_xticks(range(0, 20))
    ax.set_xticklabels([f"F-{i+1}" for i in range(20)], fontsize=9)
    ax.set_yticks([0.9, 1.9, 2.9, 3.9])
    ax.set_yticklabels(["Raf 1 (Alt)", "Raf 2", "Raf 3", "Raf 4 (Üst)"], fontsize=10, weight='bold')
    ax.grid(False)

    # Grafik Başlığı
    plt.title(f"📊 MASKELENMİŞ PLANOGRAM MİMARİSİ — {segment} SEGMENTİ [{model.upper()}]\n"
              f"Kısıt Doğrulaması: 4 Raf x 20 Facing = 80 Kapasite | 63 SKU (Minimum 1 Facing Garantisi ve Delist Yasağı Aktif)",
              fontsize=11, pad=15, weight='bold', color='#1F497D')

    # GÜNCELLENEN DOSYA ADI YAPISI (Türkçe karakterlerden arındırılmış kurumsal format)
    dosya_adi_haritası = {
        "BAKIR": "Bakir", "BRONZ": "Bronz", "GÜMÜŞ": "Gumus", "ALTIN": "Altin"
    }
    model_adi_haritası = {
        "XGBoost": "XGBoost", "Random Forest": "Random_Forest", "Gradient Boost": "Gradient_Boost"
    }

    seg_clean = dosya_adi_haritası[segment]
    mod_clean = model_adi_haritası[model]

    # Çıktı klasörü ve dosya adı
    output_dir = 'cık_10'
    os.makedirs(output_dir, exist_ok=True)
    filename = os.path.join(output_dir, f"{seg_clean}_{mod_clean}_Planogrami.png")

    plt.tight_layout()
    plt.savefig(filename, dpi=200)
    plt.close()

# 12 Kombinasyonu da yeni dosya adlarıyla diske çizdirme
print("🎨 Maskelenmiş planogram şablonları yeni dosya adlarıyla çiziliyor...")
for seg in segmentler:
    for mod in modeller:
        plot_masked_visual_planogram(seg, mod, planogram_deposu[(seg, mod)])
print("✔ 12 adet görsel planogram tasarımı güncel dosya adlarıyla (Örn: Altin_XGBoost_Planogrami.png) başarıyla kaydedildi!")

# ==============================================================================
# 3. ADIM: MASKELENMİŞ MODEL KARŞILAŞTIRMA VE DAĞILIM EXCEL RAPORU
# ==============================================================================

wb = openpyxl.Workbook()

# --- SEKME 1: MODEL KARŞILAŞTIRMASI ---
ws1 = wb.active
ws1.title = "Model Karşılaştırması"
ws1.views.sheetView[0].showGridLines = True

model_data = [
    ["Model Doğruluk Karşılaştırması — XGBoost | Random Forest | Gradient Boost", "", "", "", "", ""],
    ["SMOTE Dengeli Veri (n=410)  |  63 Maskelenmiş SKU  |  k=4 KMeans Segment Etiketleri  |  Test Seti Metrikleri", "", "", "", "", ""],
    ["METRİK", "XGBoost", "Random Forest", "Gradient Boost", "EN İYİ MODEL", "AÇIKLAMA"],
    ["SINIFLANDIRMA METRİKLERİ", "", "", "", "", ""],
    ["Accuracy", "93.10%", "94.83%", "87.93%", "Random Forest", "Doğruluk oranı (test seti)erosion"]
]

for row in model_data:
    ws1.append(row)

# Hücre birleştirmeleri
ws1.merge_cells("A1:F1")
ws1.merge_cells("A2:F2")
ws1.merge_cells("A4:F4")
ws1.merge_cells("A14:F14")

# Kurumsal Stil Tanımları
font_title = Font(name="Segoe UI", size=13, bold=True, color="FFFFFF")
font_subtitle = Font(name="Segoe UI", size=10, italic=True, color="FFFFFF")
font_header = Font(name="Segoe UI", size=10, bold=True, color="FFFFFF")
font_section = Font(name="Segoe UI", size=10, bold=True, color="1F497D")
font_bold_cell = Font(name="Segoe UI", size=10, bold=True)
font_regular = Font(name="Segoe UI", size=10)

fill_dark_blue = PatternFill(start_color="1F497D", end_color="1F497D", fill_type="solid")
fill_med_blue = PatternFill(start_color="366092", end_color="366092", fill_type="solid")
fill_light_gray = PatternFill(start_color="F2F5F8", end_color="F2F5F8", fill_type="solid")
fill_section = PatternFill(start_color="D9E1F2", end_color="D9E1F2", fill_type="solid")
fill_white = PatternFill(start_color="FFFFFF", end_color="FFFFFF", fill_type="solid")

thin_side = Side(style='thin', color='D9D9D9')
border_thin = Border(left=thin_side, right=thin_side, top=thin_side, bottom=thin_side)

ws1["A1"].font = font_title
ws1["A1"].fill = fill_dark_blue
ws1["A1"].alignment = Alignment(horizontal="center", vertical="center")
ws1.row_dimensions[1].height = 35

ws1["A2"].font = font_subtitle
ws1["A2"].fill = fill_dark_blue
ws1["A2"].alignment = Alignment(horizontal="center", vertical="center")
ws1.row_dimensions[2].height = 22

ws1.row_dimensions[3].height = 26
for c_idx in range(1, 7):
    cell = ws1.cell(row=3, column=c_idx)
    cell.font = font_header
    cell.fill = fill_med_blue
    cell.border = border_thin
    cell.alignment = Alignment(horizontal="center", vertical="center")

for r_idx in range(4, len(model_data) + 1):
    ws1.row_dimensions[r_idx].height = 20
    first_cell_val = ws1.cell(row=r_idx, column=1).value

    if first_cell_val in ["SINIFLANDIRMA METRİKLERİ", "HATA VE GÜVEN METRİKLERİ"]:
        for c_idx in range(1, 7):
            cell = ws1.cell(row=r_idx, column=c_idx)
            cell.font = font_section
            cell.fill = fill_section
            cell.border = border_thin
            cell.alignment = Alignment(horizontal="left", vertical="center")
    else:
        row_fill = fill_light_gray if r_idx % 2 == 0 else fill_white
        for c_idx in range(1, 7):
            cell = ws1.cell(row=r_idx, column=c_idx)
            cell.font = font_regular
            cell.fill = row_fill
            cell.border = border_thin
            if c_idx in [2, 3, 4, 5]:
                cell.alignment = Alignment(horizontal="center", vertical="center")
                if c_idx == 5: cell.font = font_bold_cell
            else:
                cell.alignment = Alignment(horizontal="left", vertical="center")


# --- SEKME 2: PLANOGRAM FACING DAĞILIMI (MASKELENMİŞ VERİ) ---
ws2 = wb.create_sheet(title="Planogram Facing Dağılımı")
ws2.views.sheetView[0].showGridLines = True

ws2.append(["Segment x Model Maskelenmiş Planogram Dağılım Listesi", "", "", "", "", ""])
ws2.merge_cells("A1:F1")
ws2["A1"].font = font_title
ws2["A1"].fill = fill_dark_blue
ws2["A1"].alignment = Alignment(horizontal="center", vertical="center")
ws2.row_dimensions[1].height = 35

ws2.append(["Segment", "Model", "Raf Numarası", "Raf İçi Sıralı Maskelenmiş SKU Düzeni (Her Raf Tam 20 Ürün)", "", ""])
ws2.merge_cells("D2:F2")
ws2.row_dimensions[2].height = 25
for col_idx in range(1, 5):
    cell = ws2.cell(row=2, column=col_idx)
    cell.font = font_header
    cell.fill = fill_med_blue
    cell.alignment = Alignment(horizontal="center", vertical="center")
    cell.border = border_thin

row_tracker = 3
for seg in segmentler:
    for mod in modeller:
        shelves = planogram_deposu[(seg, mod)]
        for r_num, shelf_content in enumerate(shelves):
            content_str = ", ".join(shelf_content)
            ws2.append([seg, mod, f"Raf {r_num+1}", content_str, "", ""])
            ws2.merge_cells(start_row=row_tracker, start_column=4, end_row=row_tracker, end_column=6)

            for c_idx in range(1, 7):
                cell = ws2.cell(row=row_tracker, column=c_idx)
                cell.font = font_regular
                cell.border = border_thin
                if c_idx <= 3:
                    cell.alignment = Alignment(horizontal="center", vertical="center")
                else:
                    cell.alignment = Alignment(horizontal="left", vertical="center")
            row_tracker += 1

# Dikey hücre birleştirmeleri ile gruplama tasarımı
for s_block in range(0, 4):
    start_r = 3 + (s_block * 12)
    ws2.merge_cells(start_row=start_r, start_column=1, end_row=start_r+11, end_column=1)
    ws2.cell(row=start_r, column=1).font = font_section

    for m_block in range(0, 3):
        m_start_r = start_r + (m_block * 4)
        ws2.merge_cells(start_row=m_start_r, start_column=2, end_row=m_start_r+3, end_column=2)


# --- SEKME 3: MODEL RAPOR ÖZETİ (MASKELENMİŞ DETAYLAR) ---
ws3 = wb.create_sheet(title="Model Rapor Özeti")
ws3.views.sheetView[0].showGridLines = True

comment_data = [
    ["Model Karar Ağacı Yorumları & Maskelenmiş Planogram Mantığı", "", ""],
    ["MODEL", "STRATEJİK PLANOGRAM SEÇİMİ VE ANALİZ NOTLARI (ANONİMLEŞTİRİLMİŞ)", ""],
    ["RANDOM FOREST\n(Genel Lider)", "Test kümesinde %94.83 ve Çapraz Doğrulamada %94.39 skor elde ederek en istikrarlı model seçilmiştir.\nHacimli kitle SKU'ları (örneğin M-35, M-50, M-61) bu model altındaki planogramlarda öne çıkar.", ""],
    ["XGBOOST\n(Güven Sınırı)", "Log Loss değeri 0.159 ile en başarılı olasılık kalibrasyonuna sahiptir. Sınıf sınırlarında kalan mağazaların segmentasyon geçişlerinde en güvenilir ayrımı yapar. Üst segmentlerde (ALTIN/GÜMÜŞ) yer alan stratejik niş SKU'lar (M-23, M-11) bu modelde daha hassas dağıtılır.", ""],
    ["GRADIENT BOOST\n(Destekleyici)", "Test kümesinde %87.93 ile diğerlerine göre daha düşük kalmıştır. Tek başına planogram ataması yapmaktansa ensemble mimari içi destekçi olarak konumlandırılması önerilir. Teknoloji/Trend odaklı maskeli SKU grupları (M-54, M-62) bu model yapısında görece yüksek ağırlık almaktadır.", ""]
]

for row in comment_data:
    ws3.append(row)

ws3.merge_cells("A1:C1")
ws3.merge_cells("B2:C2")
ws3["A1"].font = font_title
ws3["A1"].fill = fill_dark_blue
ws3["A1"].alignment = Alignment(horizontal="center", vertical="center")
ws3.row_dimensions[1].height = 35

for c_idx in range(1, 3):
    cell = ws3.cell(row=2, column=c_idx)
    cell.font = font_header
    cell.fill = fill_med_blue
    cell.alignment = Alignment(horizontal="center", vertical="center")
    cell.border = border_thin

for r_idx in range(3, 6):
    ws3.merge_cells(f"B{r_idx}:C{r_idx}")
    ws3.row_dimensions[r_idx].height = 65
    row_fill = fill_light_gray if r_idx % 2 == 0 else fill_white
    for c_idx in range(1, 4):
        cell = ws3.cell(row=r_idx, column=c_idx)
        cell.font = font_regular
        cell.fill = row_fill
        cell.border = border_thin
        cell.alignment = Alignment(horizontal="left", vertical="center", wrap_text=True)
    ws3.cell(row=r_idx, column=1).font = font_bold_cell
    ws3.cell(row=r_idx, column=1).alignment = Alignment(horizontal="center", vertical="center")

# Sütun Genişliklerini Otomatik Ayarlama
for ws in [ws1, ws2, ws3]:
    for col in ws.columns:
        max_len = 0
        col_letter = get_column_letter(col[0].column)
        for cell in col:
            if cell.row == 1: continue
            if cell.value:
                sub_lines = str(cell.value).split('\n')
                for line in sub_lines:
                    if len(line) > max_len: max_len = len(line)
        ws.column_dimensions[col_letter].width = max(max_len + 4, 13)

# Excel Dosyasını Kaydetme
excel_output_name = "Model_Karsilastirma_Maskeli_Planogram_Raporu.xlsx"
wb.save(excel_output_name)
print(f"✔ Maskelenmiş kurumsal analiz raporu '{excel_output_name}' adıyla başarıyla üretildi!")
